# Tugas Praktikum: Model Machine Learning Menggunakan Naive Bayes
Tujuan: Memprediksi Tingkat Literasi AI (Perilaku Cek Fakta).

**Kelompok 4**
- Fizar Erlansyah – 2024081041
- Angel Claudia Pakpahan - 2025011055

## Tahap Persiapan & Pemuatan Data
Langkah pertama adalah mengimpor library, dan memastikan set data Excel/CSV sudah dapat terbaca.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

file_name = "🤖 Survei Penggunaan AI_ Tren Adopsi di Kalangan Mahasiswa & Profesional 🚀  (Jawaban) - Form Responses 1.csv"
df = pd.read_csv(file_name)
print(f"✅ Data berhasil dimuat. Total baris dataset: {df.shape[0]}, Total kolom: {df.shape[1]}")

## Data Cleaning (Pembersihan Variasi Teks)
Dikarenakan pengisian nama universitas berbentuk *Free Text*, seringkali peserta melakukan *typo* huruf atau penyingkatan (contoh "Upj" atau "universitas pamulanb"). Agar laporan kita valid dan bagus, kita akan menstandarisasi penulisan tersebut sebelum digunakan. Berikut adalah perbandingan sebelum dan sesudahnya.

In [ ]:
univ_col = "1. Asal Universitas (Nama Lengkap Universitas)"
nama_col = "Nama Lengkap  "

# 1. Menyimpan data sebelum dibersihkan (Untuk visualisasi komparasi)
df_before = df[[nama_col, univ_col]].copy()

# 2. Fungsi Standardisasi Kata
def clean_univ(name):
    if pd.isna(name):
        return "-"
    name = str(name).strip().upper()
    
    if "UPJ" in name or "PEMBANGUNAN JAYA" in name:
        return "Universitas Pembangunan Jaya"
    elif "PAMULAN" in name:
        return "Universitas Pamulang"
    elif "TIRTAYASA" in name:
        return "Universitas Sultan Ageng Tirtayasa"
    else:
        return name.title()

# 3. Menerapkan pembersihan
df[univ_col] = df[univ_col].apply(clean_univ)
df[nama_col] = df[nama_col].fillna("Anonim")

# 4. Menampilkan Hasil Visual Tabel Pre & Post Data Cleaning khusus yang terdampak typo
df_compare = pd.DataFrame({
    'Nama Lengkap': df[nama_col],
    'Universitas Asli (Kotor)': df_before[univ_col],
    'Universitas Bersih (Standar)': df[univ_col]
})

# Tampilkan hanya baris yang tulisan aslinya berbeda dari format standar agar kelihatan perubahannya
df_changed = df_compare[df_compare['Universitas Asli (Kotor)'] != df_compare['Universitas Bersih (Standar)']]

print("✅ Pembersihan string teks selesai. Berikut adalah rekapan tabel Universitas yang dikoreksi otomatis:\n")
display(df_changed)

## Preprocessing Model Machine Learning
Tahapan preparasi AI meliputi:
1. Imputasi Modus & Median (Menghindari Error Data Kosong)
2. Label Binarisasi (Mengubah output 1-5 menjadi "Kritis" dan "Percaya Buta")
3. Matrix Encoding (Mengubah kategori teks menjadi matriks nilai agar sanggup difahami oleh probabilitas matematika Bayes)

In [ ]:
target_col = "5. Seberapa sering Anda melakukan pengecekan ulang (fakta/logika) terhadap hasil yang diberikan oleh AI? 🔍"
cat_feature_1 = "Status Utama Anda Saat Ini 💼  "
num_feature_1 = "1. Seberapa paham Anda mengenai teknik penulisan instruksi (Prompt Engineering) untuk AI? 🧠"
cat_feature_2 = "4. Jika dinilai, masuk ke dalam klasifikasi manakah tingkat ketergantungan Anda terhadap AI? (Pilih satu yang paling menggambarkan Anda) 📊"

features = [cat_feature_1, num_feature_1, cat_feature_2]
df_subset = df[features + [target_col]].copy()

# A. Penanganan Missing Values
for col in features:
    if pd.api.types.is_numeric_dtype(df_subset[col]):
        df_subset[col] = df_subset[col].fillna(df_subset[col].median())
    else:
        if not df_subset[col].mode().empty:
            df_subset[col] = df_subset[col].fillna(df_subset[col].mode()[0])
        
if pd.api.types.is_numeric_dtype(df_subset[target_col]):
    df_subset[target_col] = df_subset[target_col].fillna(df_subset[target_col].median())
else:
    if not df_subset[target_col].mode().empty:
        df_subset[target_col] = df_subset[target_col].fillna(df_subset[target_col].mode()[0])

# B. Transformasi Target Binarisasi
def transform_target(value):
    return "Kritis" if str(value).isdigit() and int(value) >= 4 else "Percaya Buta"

df_subset['Target'] = df_subset[target_col].apply(transform_target)
X = df_subset[features]
y = df_subset['Target']

# C. Encoding Fitur Kategorik dan Label
le = LabelEncoder()
y_encoded = le.fit_transform(y)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), [cat_feature_1, cat_feature_2]),
        ('num', 'passthrough', [num_feature_1])
    ]
)

X_processed = preprocessor.fit_transform(X)
print("✅ Fitur X (Matriks) dan Label Y (Target) siap di-training.\n")

## Aktivitas 2: Analisis Kasus & Prediksi AI
Kami akan membagi dataset menjadi Porsi Training dan secara eksplisit menyisihkan **10 sampel observasi Testing** sesuai mandat instruksi Aktivitas 2.

In [ ]:
# Trik Khusus: Kita ambil index orisinil-nya juga agar entitas Nama Mahasiswa & Univ bisa ditarik kembali pada df_hasil
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_processed, y_encoded, df_subset.index, test_size=10, random_state=42
)
print(f"✅ Split Data Selesai: {len(y_train)} Data Training, {len(y_test)} Data Pengujian (Testing).\n")

# Pelatihan Model Classifier Naive Bayes
model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test) # Ekstraksi probabilitas untuk keyakinan matematis

# -----------------------------------------------------
# PEMBUATAN TABEL LAPORAN (Menampilkan Detail Mahasiswa + Akurasi)
# -----------------------------------------------------

# 1. Ekstrak Kolom Identitas dari barisan testing 
df_identitas = df.loc[idx_test, [nama_col, univ_col]].reset_index(drop=True)
df_identitas.rename(columns={ nama_col: "Nama Peserta", univ_col: "Universitas" }, inplace=True)

# 2. Ekstrak Kolom Fitur Asli dari barisan testing
df_fitur = df_subset.loc[idx_test, features].reset_index(drop=True)
df_fitur.rename(columns={
    cat_feature_1: "Status Pekerjaan",
    num_feature_1: "Skor Prompt",
    cat_feature_2: "Ketergantungan AI"
}, inplace=True)

# 3. Gabungkan seluruh kolom (Concat) agar sangat representatif ketika dipajang di Word Mhs.
df_hasil = pd.concat([df_identitas, df_fitur], axis=1)
df_hasil['Sifat Aktual (Asli)'] = le.inverse_transform(y_test)
df_hasil['Tebakan Prediksi AI'] = le.inverse_transform(y_pred)
df_hasil['Probabilitas Keyakinan'] = [f"{max(proba)*100:.1f}%" for proba in y_proba]
df_hasil['Jawaban Tepat?'] = df_hasil['Sifat Aktual (Asli)'] == df_hasil['Tebakan Prediksi AI']

# Menerbitkan DataFrame Akhir
display(df_hasil)

# -----------------------------------------------------
# KALKULASI HASIL NUMERIK 
# -----------------------------------------------------
jumlah_benar = (y_test == y_pred).sum()
jumlah_total = len(y_test)
print(f"\n👉 Analisis Jumlah Tepat: AI berhasil menebak {jumlah_benar} dari total {jumlah_total} data tes pengujian.")
akurasi = accuracy_score(y_test, y_pred)
print(f"👉 Rasio Akurasi Keseluruhan: {akurasi * 100:.2f}%\n")

# Visualisasi Matriks Kebingungan (Confusion Matrix)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 3.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix - Literasi AI Mahasiswa', fontsize=12)
plt.xlabel('Prediksi Model', fontsize=10)
plt.ylabel('Sifat Aktual Sesungguhnya', fontsize=10)
plt.show()